In [14]:
import cutlass 
import cutlass.cute as cute 
import torch
import numpy as np
from cutlass.torch import dtype as torch_dtype
import cutlass.cute.runtime as cute_rt
from cutlass.cute.runtime import from_dlpack


A tensor in cute, is essentially an engine (which is  a random access iterator) with a start offset $E$ and a layout $L = S:D$ and a value map from the engine offsets to the value space (whatever dtype and so on) 

indeed, the layout map of a coordinate tuple $X$, $L(X)$ is such that, the access of tensor T at co-ordinate $X$, $T[X]$ is the same as getting the value map at $E + L(X)$
that is a tensor $T = E \circ L$ where the action of composition of the Engine does $x \mapsto E_{offset} + L(x)$ where $x$ is the access co-ordinate. 

Below I pute CuteDSL's own explaination as its ON POINT. 

Tensor
A tensor in CuTe is created through the composition of two key components:

An Engine (E) - A random-access, pointer-like object that supports:

Offset operation: e + d → e (offset engine by elements of a layout's codomain)
Dereference operation: *e → v (dereference engine to produce value)
A Layout (L) - Defines the mapping from coordinates to offsets

A tensor is formally defined as the composition of an engine E with a layout L, expressed as T = E ∘ L. When evaluating a tensor at coordinate c, it:

Maps the coordinate c to the codomain using the layout
Offsets the engine accordingly
Dereferences the result to obtain the tensor's value
This can be expressed mathematically as:

T(c) = (E ∘ L)(c) = *(E + L(c))


To that end, slicing, (in cute you can either pick the co-ordinate for a dim or take the full slice of that dim, no numpy [a:b,c:d, x,y] type shit, cause look one can simply tile a layout to get another tiled layout and if you want tile slices you can slice the inner modes and so on)

okay so slicing: what's the deal with slicing, well, I can express my slice as co-ordinate X = (x0,x1,...x_dim-1) WHERE $x_i \in [0,s_i) \cup {NONE}$ 
The idea then is simple, we can filter entries of the layout $L$ such that the new sliced layout $L'= S':D'$ collects modes of $L$ for which the slice co-ordinate was None (its kinda odd that taking the whole dim is called NONE). 

and for the engine, well you get a new engine, which is already offset by the layout image of the co-ordinates that were not NONE, setting the None co-ordinates to zero. 

>Restrict operator 
>Let $L = S:D$, and $I \subset [0, \text{rank}(L))$ then obviously, $I$ induces a sub-sequence of the modes, and taking only those modes, we get $L^{|I} = S^{|I}: D^{|I}$.  

given the  above restrict operator we can set $I = \{i: x_i = \text{NONE}\}$ and use it to get $L' = L^{|I}$
where $L'$ is the sliced layout, and we can also make the "offset-co-ordinate" 
X_off = <X_i if X_i is not NONE else 0>
and then the new engine start is $E' = E + L(X_off)$ 
and the new sliced tensor is $T' = E' \circ L'$


The simplest way to make a cute.Tensor is to use the from_dl_pack on torch/numpy tensors, it will inherit the layout of those tensors. you can also see what kind of engine is being composed with the layout, below it will show that the engine is:

ptr<bf16,generic> 

that means that the memory space is generic. 

There are many memory spaces (I am not sure if DSMEM is a seperate memory space here btw we will find out when we write for hopper)

- **generic**: Default memory space that can refer to any other memory space.
- **global memory (gmem)**: Accessible by all threads across all blocks, but has higher latency.
- **shared memory (smem)**: Accessible by all threads within a block, with much lower latency than global memory.
- **register memory (rmem)**: Thread-private memory with the lowest latency, but limited capacity.
- **tensor memory (tmem)**: Specialized memory introduced in NVIDIA Blackwell architecture for tensor operations.

Also another note, cute.print_tensor seems to only work when the tensor resides on the CPU (the chosen mem space will be generic) 

as the below cell will throw error if I use device = "cuda" 




In [15]:
@cute.jit
def print_tensor_dlpack(src: cute.Tensor):
    print(src)
    cute.print_tensor(src)
    cute.printf(src.layout)
    

T = torch.randn(4,3, dtype=torch_dtype(cutlass.BFloat16), device = "cpu").transpose(0,1)
print_tensor_dlpack(from_dlpack(T))

tensor<ptr<bf16, generic> o (3,4):(1,3)>
tensor(raw_ptr(0x000059a067f52e40: bf16, generic, align<2>) o (3,4):(1,3), data=
       [[-0.726562, -1.031250,  0.890625,  0.079102, ],
        [-1.804688, -1.257812, -0.490234, -0.232422, ],
        [ 1.421875, -2.562500, -1.210938, -0.087402, ]])
(3,4):(1,3)


In [16]:
@cute.jit
def create_tensor_from_ptr(ptr: cute.Pointer):
    layout = cute.make_layout((8, 5), stride=(5, 1))
    tensor = cute.make_tensor(ptr, layout)
    tensor.fill(1)
    cute.print_tensor(tensor)
  

x = torch.empty((40,),dtype=torch_dtype(cutlass.BFloat16),device='cpu')
print(x)

create_tensor_from_ptr(cute_rt.make_ptr(cutlass.BFloat16, x.data_ptr()))
print(x)

tensor([ 7.0615e+36, -2.7711e-23,  4.5918e-40,  0.0000e+00,  0.0000e+00,
         0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
         0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
         0.0000e+00,  0.0000e+00,  0.0000e+00,  2.9387e-39,  0.0000e+00,
         2.0896e+23,  2.0407e+20,  5.2536e+22,  1.0361e-08,  6.2937e-10,
         4.2608e-08,  1.2790e-11,  2.1251e+23,  3.3975e-06,  4.1246e-05,
         2.1327e-07,  1.2790e-11,  6.5120e-10,  5.2241e+22,  2.6517e+20,
         3.2466e+21,  2.6681e+23,  3.3573e+21,  0.0000e+00,  0.0000e+00],
       dtype=torch.bfloat16)
tensor(raw_ptr(0x000059a067caa080: bf16, generic, align<2>) o (8,5):(5,1), data=
       [[ 1.000000,  1.000000,  1.000000,  1.000000,  1.000000, ],
        [ 1.000000,  1.000000,  1.000000,  1.000000,  1.000000, ],
        [ 1.000000,  1.000000,  1.000000,  1.000000,  1.000000, ],
        ...
        [ 1.000000,  1.000000,  1.000000,  1.000000,  1.000000, ],
        [ 1.000000,  1.00

In [17]:
@cute.jit
def test(mX: cute.Tensor):
    cute.printf("\nGlobal tensor mX: {}", mX)
    idX = cute.make_identity_tensor(mX.shape)
    cute.printf("\nIdentity tensor: {}", idX)


x = np.random.rand(14 * 1024).reshape((14, 1024))
test(from_dlpack(x))


Global tensor mX: raw_ptr(0x000059a068f023c0: f64, generic, align<8>) o (14,1024):(1024,1) = 
  ( 0.446934, 0.721871, 0.245871, 0.207931, 0.938864, 0.926520, 0.436597, 0.550464, 0.037575, 0.555116, 0.611238, 0.044314, 0.962998, 0.772718, 0.411518, 0.531374, 0.503415, 0.602979, 0.146478, 0.662544, 0.770413, 0.981279, 0.243507, 0.742277, 0.496684, 0.583998, 0.494167, 0.934883, 0.307296, 0.496597, 0.516420, [...] )

Identity tensor: (0,0) o (14,1024):(1@0,1@1)


In [18]:
@cute.jit
def make_coordinate_tensor_2d(shape):
  layout = cute.make_layout(shape, stride=(cute.ScaledBasis(1,0),cute.ScaledBasis(1,1))) 
  
  tensor = cute.make_tensor((0,0),layout)
  cute.print_tensor(tensor)
  
  
make_coordinate_tensor_2d((4,3))



tensor((0,0) o (4,3):(1@0,1@1), data=
       [[ (0,0),  (0,1),  (0,2), ],
        [ (1,0),  (1,1),  (1,2), ],
        [ (2,0),  (2,1),  (2,2), ],
        [ (3,0),  (3,1),  (3,2), ]])


Indeed, the idea is simple, and we can ispect the needed stride for co-ordinate tensors of particular shapes by using the make identity tensor function cause as usual the obvious ideas are WRONG 

In [76]:
@cute.jit
def make_identity_tensor(shape): 
  tensor = cute.make_identity_tensor(shape)
  cute.printf(tensor.layout)
  cute.print_tensor(tensor)
  
  
make_identity_tensor((4,(2,3)))

(4,(2,3)):(1@0,(1@0@1,1@1@1))
tensor((0,(0,0)) o (4,(2,3)):(1@0,(1@0@1,1@1@1)), data=
       [[ (0,(0,0)),  (0,(1,0)),  (0,(0,1)),  (0,(1,1)),  (0,(0,2)),  (0,(1,2)), ],
        [ (1,(0,0)),  (1,(1,0)),  (1,(0,1)),  (1,(1,1)),  (1,(0,2)),  (1,(1,2)), ],
        [ (2,(0,0)),  (2,(1,0)),  (2,(0,1)),  (2,(1,1)),  (2,(0,2)),  (2,(1,2)), ],
        [ (3,(0,0)),  (3,(1,0)),  (3,(0,1)),  (3,(1,1)),  (3,(0,2)),  (3,(1,2)), ]])


In [ ]:
@cute.jit
def foo(): 
  cute.print()

In [101]:
@cute.jit 
def make_scaled_tensor(shape): 
  d0 = cute.ScaledBasis(1,[0])
  d1 = cute.ScaledBasis(1,[1,0])
  d2 = cute.ScaledBasis(1,[1,1])
  stride = (d0,(d1,d2))
  cute.printf(stride)
  layout = cute.make_layout(shape, stride=stride)
  tensor = cute.make_tensor((0,0),layout)
  cute.print_tensor(tensor)

In [102]:
make_scaled_tensor((4,(2,3)))

(1@0,(1@0@1,1@1@1))
tensor((0,0) o (4,(2,3)):(1@0,(1@0@1,1@1@1)), data=
       [[ (0,(0,0)),  (0,(1,0)),  (0,(0,1)),  (0,(1,1)),  (0,(0,2)),  (0,(1,2)), ],
        [ (1,(0,0)),  (1,(1,0)),  (1,(0,1)),  (1,(1,1)),  (1,(0,2)),  (1,(1,2)), ],
        [ (2,(0,0)),  (2,(1,0)),  (2,(0,1)),  (2,(1,1)),  (2,(0,2)),  (2,(1,2)), ],
        [ (3,(0,0)),  (3,(1,0)),  (3,(0,1)),  (3,(1,1)),  (3,(0,2)),  (3,(1,2)), ]])


Okay, so its pretty obvious that natural number strides can be replaced by N^m "semi-vectors" (dont recall the name it does form a ring type shit or a module i guess is what its called is a vector space without inverses cause N^m and elementwise add got no invesrse in N^m nvm why am I going on this tangent) 
well, cause then your range/co-domain isn't gonna be natural numbers its gonna be N^m or even N^m x (N^p x N^q) *(nested bracketing sitaution)

and still if you have some "scaled basis vectors" D0,D1,D2,...DM then an m-coordinate x0,x1,x2..xm can be taken to the co-domain co-ordinate space by doing the linear combination x0DO + x1D1 + ... xmDm (of course this asssumes that each D_i has the same profile) 

and to give thse basis vectors 
cute gives us the very intutive and simple to understand cute.ScaledBasis(a,[b0,b1,b2,b3]) which obviously means 
take the number a, put it in position b3, 
take the resulting basis vector and put it in position b2, 
take the then resulting vector and put it in poisition b1, 
take the yet again resulting vector and put in position b0, 

so if we think about this, we are going (0,0,...,(0,0,...,0,(0,0,...,0,(0,..a@b3,...)@b2,0,0,...0)@b1,0,...,0)@b0,0,...0)
which means that our profile has a depth of 4. 

that is it is read as ((((a@b3)@b2)@b1)@b0)


like even if this shit isn't perfect in our head (and trust me it wont be for me the convention is goofy imo) we can just print the identity tensors to see what basis we want. 
  

In [103]:
make_identity_tensor((3,(4,5,(6,7,8),(9,10))))

(3,(4,5,(6,7,8),(9,10))):(1@0,(1@0@1,1@1@1,(1@0@2@1,1@1@2@1,1@2@2@1),(1@0@3@1,1@1@3@1)))
tensor((0,(0,0,(0,0,0),(0,0))) o (3,(4,5,(6,7,8),(9,10))):(1@0,(1@0@1,1@1@1,(1@0@2@1,1@1@2@1,1@2@2@1),(1@0@3@1,1@1@3@1))), data=
       [[ (0,(0,0,(0,0,0),(0,0))),  (0,(1,0,(0,0,0),(0,0))),  (0,(2,0,(0,0,0),(0,0))), ...,  (0,(1,4,(5,6,7),(8,9))),  (0,(2,4,(5,6,7),(8,9))),  (0,(3,4,(5,6,7),(8,9))), ],
        [ (1,(0,0,(0,0,0),(0,0))),  (1,(1,0,(0,0,0),(0,0))),  (1,(2,0,(0,0,0),(0,0))), ...,  (1,(1,4,(5,6,7),(8,9))),  (1,(2,4,(5,6,7),(8,9))),  (1,(3,4,(5,6,7),(8,9))), ],
        [ (2,(0,0,(0,0,0),(0,0))),  (2,(1,0,(0,0,0),(0,0))),  (2,(2,0,(0,0,0),(0,0))), ...,  (2,(1,4,(5,6,7),(8,9))),  (2,(2,4,(5,6,7),(8,9))),  (2,(3,4,(5,6,7),(8,9))), ]])
